# Heart Failure Detection Demo

This notebook demonstrates the heart failure detection pipeline.

In [ ]:
# Import necessary libraries
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

# Import project modules
from src.data_processing.data_loader import load_dataset
from src.data_processing.preprocessor import DataPreprocessor
from src.model_development.model_trainer import ModelTrainer
from src.visualization.data_visualizer import DataVisualizer
from src.visualization.model_visualizer import ModelVisualizer

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

## 1. Load the Dataset

In [ ]:
# Load the dataset
data = load_dataset()

# Display the first few rows
data.head()

## 2. Explore the Dataset

In [ ]:
# Display basic information about the dataset
print(f"Dataset shape: {data.shape}")
print(f"\nData types:\n{data.dtypes}")
print(f"\nMissing values:\n{data.isnull().sum()}")

In [ ]:
# Display summary statistics
data.describe(include=[np.number]).T

## 3. Visualize the Data

In [ ]:
# Initialize data visualizer
visualizer = DataVisualizer()

# Plot target distribution
visualizer.plot_target_distribution(data)

In [ ]:
# Plot correlation matrix
visualizer.plot_correlation_matrix(data)

In [ ]:
# Plot feature distributions by target
visualizer.plot_feature_distributions_by_target(data, top_n=5)

## 4. Preprocess the Data

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor()

# Preprocess data
X_train, X_test, y_train, y_test = preprocessor.preprocess_data(data, save_transformers=False)

# Display preprocessed data
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

## 5. Train and Evaluate Models

In [ ]:
# Initialize model trainer
trainer = ModelTrainer()

# Train and evaluate models
results_df = trainer.train_and_evaluate_models(X_train, X_test, y_train, y_test)

# Display results
results_df

In [ ]:
# Initialize model visualizer
model_visualizer = ModelVisualizer()

# Plot model comparison
model_visualizer.plot_model_comparison(results_df)

## 6. Train and Evaluate Best Model

In [ ]:
# Train best model
best_model = trainer.train_best_model(X_train, y_train)

# Evaluate best model
evaluation = trainer.evaluate_model(best_model, X_test, y_test)

# Display evaluation results
print(f"Accuracy: {evaluation['accuracy']:.4f}")
print(f"ROC AUC: {evaluation['roc_auc']:.4f}" if evaluation['roc_auc'] else "ROC AUC: Not available")
print(f"\nClassification Report:\n{pd.DataFrame(evaluation['classification_report']).T}")

In [ ]:
# Plot confusion matrix
y_pred = best_model.predict(X_test)
model_visualizer.plot_confusion_matrix(y_test, y_pred)

In [ ]:
# Plot ROC curve if model supports predict_proba
if hasattr(best_model, 'predict_proba'):
    y_pred_proba = best_model.predict_proba(X_test)[:, 1]
    model_visualizer.plot_roc_curve(y_test, y_pred_proba)

In [ ]:
# Plot feature importance if model supports it
if hasattr(best_model, 'feature_importances_'):
    model_visualizer.plot_feature_importance(best_model, X_train.columns)

## 7. Save the Best Model

In [ ]:
# Save the best model
save_success = trainer.save_model(best_model)
print(f"Model saved successfully: {save_success}")

## 8. Make Predictions with the Model

In [ ]:
# Create a sample input
sample_input = pd.DataFrame({
    'Age': [65],
    'Sex': ['Male'],
    'NYHA': ['II'],
    'HTN': ['Yes'],
    'DM': ['Yes'],
    'Smoker': ['Yes'],
    'DL': ['Yes'],
    'BA': ['No'],
    'CXR': ['Yes'],
    'RWMA': ['Yes'],
    'MI': ['Yes'],
    'Chest_pain': ['Yes'],
    'ECG': ['Abnormal'],
    'ACS': ['STEMI'],
    'Wall': ['Anterior'],
    'MR': ['Moderate'],
    'Thrombolysis': ['Yes'],
    'EF': [35],
    'HR': [95],
    'SBP': [140],
    'DBP': [90],
    'Creatinine': [1.5],
    'Sodium': [135],
    'Potassium': [4.2]
})

# Preprocess the input
preprocessed_input = preprocessor.preprocess_new_data(sample_input)

# Make prediction
prediction = best_model.predict(preprocessed_input)[0]

# Get prediction probability if available
if hasattr(best_model, 'predict_proba'):
    probability = best_model.predict_proba(preprocessed_input)[0][1]
    print(f"Prediction: {prediction} (Heart Failure)" if prediction == 1 else f"Prediction: {prediction} (No Heart Failure)")
    print(f"Probability: {probability:.4f}")
else:
    print(f"Prediction: {prediction} (Heart Failure)" if prediction == 1 else f"Prediction: {prediction} (No Heart Failure)")